# Sleep Stage Baseline

Baseline based on `ref/final-hack-iot-tcnlstm-cnn-superai-ss5.ipynb`.

The reference notebook uses 480-row windows, FFT magnitude features, `StandardScaler`, weighted F1, and a CNN-LSTM model. This notebook keeps that baseline structure, with cleaner path detection, grouped validation by file, training-scaler reuse for test inference, and submission generation.

## Setup

In [ ]:
import os
import random
from collections import Counter
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import Conv1D, Dense, Dropout, Input, LSTM, MaxPooling1D
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
WINDOW_SIZE = 480  # 30 seconds * 16 Hz
SIGNAL_COLUMNS = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
LABEL_TO_ID = {"W": 0, "R": 1, "N1": 2, "N2": 3, "N3": 4}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}


def find_competition_root():
    candidates = [
        Path("/kaggle/input/Individual-Sleep-Stage-Classification"),
        Path("/kaggle/input/spai-signal-sleep-staging-classification"),
        Path("../../dataset/Individual-Sleep-Stage-Classification"),
        Path("../../dataset/Individual-Sleep-Stage-Classification"),
    ]
    for root in candidates:
        if root.exists():
            return root
    raise FileNotFoundError("Competition data root not found. Update COMPETITION_ROOT manually.")


COMPETITION_ROOT = find_competition_root()
TRAIN_DIR = COMPETITION_ROOT / "train" / "train"
TEST_DIR = COMPETITION_ROOT / "test_segment" / "test_segment"
SAMPLE_SUBMISSION = COMPETITION_ROOT / "sample_submission.csv"

print("root:", COMPETITION_ROOT)
print("train:", TRAIN_DIR)
print("test:", TEST_DIR)
print("sample submission:", SAMPLE_SUBMISSION)

## Load Files

In [ ]:
train_files = sorted(TRAIN_DIR.glob("*.csv"))
test_files = sorted(TEST_DIR.glob("**/*.csv"))

print(f"train files: {len(train_files)}")
print(f"test segment files: {len(test_files)}")

if train_files:
    preview = pd.read_csv(train_files[0], nrows=3)
    print("first train file:", train_files[0].name)
    display(preview)

if test_files:
    preview = pd.read_csv(test_files[0], nrows=3)
    print("first test file:", test_files[0].name)
    display(preview)

## Reference-Style FFT Preprocessing

In [ ]:
def clean_signal_frame(df):
    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]
    missing = [col for col in SIGNAL_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing signal columns: {missing}")

    signals = df[SIGNAL_COLUMNS].apply(pd.to_numeric, errors="coerce")
    signals = signals.interpolate(limit_direction="both").ffill().bfill().fillna(0.0)
    return signals.astype("float32")


def majority_label(window_labels):
    counts = np.bincount(window_labels, minlength=len(LABEL_TO_ID))
    return int(counts.argmax())


def preprocess_files_with_fft(files, is_training=True, window_size=WINDOW_SIZE):
    feature_segments = []
    label_segments = []
    groups = []
    ids = []

    for file_idx, file_path in enumerate(tqdm(files, desc="Processing files")):
        df = pd.read_csv(file_path)
        signals = clean_signal_frame(df)
        num_windows = len(signals) // window_size
        if num_windows == 0:
            continue

        values = signals.iloc[: num_windows * window_size].to_numpy(dtype="float32")
        windows = values.reshape(num_windows, window_size, len(SIGNAL_COLUMNS))

        # Matches the reference notebook: FFT along each 480-row window, then magnitude.
        fft_windows = np.abs(np.fft.fft(windows, axis=1)).astype("float32")
        feature_segments.append(fft_windows)
        groups.extend([file_idx] * num_windows)

        if is_training:
            if "Sleep_Stage" not in df.columns:
                raise ValueError(f"Sleep_Stage missing in {file_path}")
            labels = df["Sleep_Stage"].map(LABEL_TO_ID).to_numpy()
            labels = labels[: num_windows * window_size].reshape(num_windows, window_size)
            label_segments.append(np.apply_along_axis(majority_label, axis=1, arr=labels))
        else:
            # Test files are already one 30-second segment. Keep this generic anyway.
            stem = file_path.stem
            ids.extend([stem] if num_windows == 1 else [f"{stem}_{i:05d}" for i in range(num_windows)])

    X = np.vstack(feature_segments).astype("float32")
    groups = np.asarray(groups)
    if is_training:
        y = np.concatenate(label_segments).astype("int64")
        return X, y, groups
    return X, ids, groups

In [ ]:
X, y, groups = preprocess_files_with_fft(train_files, is_training=True)

print("X:", X.shape)
print("y:", y.shape)
print("groups:", groups.shape)
print("label counts:", {ID_TO_LABEL[k]: v for k, v in Counter(y).items()})

## Grouped Validation Split and Scaling

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, valid_idx = next(splitter.split(X, y, groups=groups))

X_train, X_valid = X[train_idx], X[valid_idx]
y_train, y_valid = y[train_idx], y[valid_idx]

scaler = StandardScaler()
train_shape = X_train.shape
valid_shape = X_valid.shape

X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(train_shape).astype("float32")
X_valid = scaler.transform(X_valid.reshape(-1, X_valid.shape[-1])).reshape(valid_shape).astype("float32")

print("train:", X_train.shape, Counter(y_train))
print("valid:", X_valid.shape, Counter(y_valid))

## Weighted F1 Metric

In [ ]:
def f1_weighted(y_true, y_pred):
    y_true = K.cast(K.reshape(y_true, (-1,)), "int64")
    y_pred_labels = K.cast(K.argmax(y_pred, axis=-1), "int64")
    num_classes = K.int_shape(y_pred)[-1]

    f1_scores = []
    weights = []
    for class_id in range(num_classes):
        true_mask = K.cast(K.equal(y_true, class_id), "float32")
        pred_mask = K.cast(K.equal(y_pred_labels, class_id), "float32")

        tp = K.sum(true_mask * pred_mask)
        fp = K.sum(pred_mask) - tp
        fn = K.sum(true_mask) - tp

        precision = tp / (tp + fp + K.epsilon())
        recall = tp / (tp + fn + K.epsilon())
        f1 = 2.0 * precision * recall / (precision + recall + K.epsilon())
        f1_scores.append(f1)
        weights.append(K.sum(true_mask))

    f1_scores = K.stack(f1_scores)
    weights = K.stack(weights)
    weights = weights / (K.sum(weights) + K.epsilon())
    return K.sum(f1_scores * weights)

## CNN-LSTM Baseline

This is the baseline model from the reference notebook, with a lower learning rate and validation-F1 callbacks.

In [ ]:
model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])),
    Conv1D(filters=128, kernel_size=3, activation="relu", padding="same"),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    Conv1D(filters=256, kernel_size=3, activation="relu", padding="same"),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(128, return_sequences=False),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(len(LABEL_TO_ID), activation="softmax"),
])

model.compile(
    loss=SparseCategoricalCrossentropy(),
    optimizer=Adam(learning_rate=1e-3),
    metrics=[f1_weighted],
)

model.summary()

In [ ]:
checkpoint = ModelCheckpoint(
    "sleep_stage_cnn_lstm_baseline.keras",
    monitor="val_f1_weighted",
    mode="max",
    save_best_only=True,
    verbose=1,
)
early_stopping = EarlyStopping(
    monitor="val_f1_weighted",
    mode="max",
    patience=10,
    restore_best_weights=True,
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_f1_weighted",
    mode="max",
    factor=0.5,
    patience=4,
    min_lr=1e-5,
    verbose=1,
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_valid, y_valid),
    epochs=40,
    batch_size=64,
    callbacks=[checkpoint, early_stopping, reduce_lr],
    verbose=2,
)

## Validation Score

In [ ]:
valid_proba = model.predict(X_valid, batch_size=256)
valid_pred = valid_proba.argmax(axis=1)

score = f1_score(y_valid, valid_pred, average="weighted")
print("weighted F1:", score)
print(classification_report(
    y_valid,
    valid_pred,
    labels=list(ID_TO_LABEL),
    target_names=[ID_TO_LABEL[i] for i in sorted(ID_TO_LABEL)],
    zero_division=0,
))

## Predict Test and Create Submission

In [ ]:
X_test, test_ids, _ = preprocess_files_with_fft(test_files, is_training=False)
test_shape = X_test.shape
X_test = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(test_shape).astype("float32")

test_proba = model.predict(X_test, batch_size=256)
test_pred = test_proba.argmax(axis=1)
test_labels = [ID_TO_LABEL[int(class_id)] for class_id in test_pred]

submission = pd.DataFrame({"id": test_ids, "labels": test_labels})

if SAMPLE_SUBMISSION.exists():
    sample = pd.read_csv(SAMPLE_SUBMISSION)
    submission = sample[["id"]].merge(submission, on="id", how="left")
    submission["labels"] = submission["labels"].fillna("W")

submission.to_csv("submission_cnn_lstm_baseline.csv", index=False)
submission.head()

In [ ]:
submission["labels"].value_counts()